In [20]:
import outlines
from enum import Enum
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "openai-community/gpt2"
hf_model = AutoModelForCausalLM.from_pretrained(model_name, device_map = "auto")
hf_tokenizer = AutoTokenizer.from_pretrained(model_name, device_map = "auto")
model = outlines.from_transformers(hf_model, hf_tokenizer)
class Conclusion(Enum):
    yes = "yes"
    no = "no"
    maybe = "maybe"
DATASET_CONTEXT = """
PubMedQA is a dataset and a task that involves Question Answering (QA) using scientific literature from PubMed, which is a free resource that contains millions of articles related to life sciences and biomedical research. PubMedQA specifically focuses on using abstracts and passages from PubMed articles to answer medical and scientific questions.
"""

In [25]:
DATASET_CONTEXT = """
PubMedQA is a dataset and a task that involves Question Answering (QA) using scientific 
literature from PubMed, which is a free resource that contains millions of articles 
related to life sciences and biomedical research. PubMedQA specifically focuses on 
using abstracts and passages from PubMed articles to answer medical and scientific questions.
"""

# — your example payload —
pubmedqa_example = {
  "contexts": [
    "Figures from the British Defence Dental Services reveal that serving personnel...",
    "Diagnostic criteria were developed...",
    "Data for 432 participants were entered into the analysis..."
  ],
  "labels": ["BACKGROUND AND AIM", "METHOD", "RESULTS"],
  "meshes": ["Adolescent", "Cross-Sectional Studies", "..."],
  "reasoning_free_pred": ["y","e","s"],
  "reasoning_required_pred": ["y","e","s"]
}

question    = "Is there a differential in the dental health of new recruits to the British Armed Forces?"
long_answer = (
    "A significant difference in dental health between recruits to each Service does exist "
    "and is likely to be a reflection of the sociodemographic background from which they are drawn."
)

# — build the chat template —
chat_messages = [
    {"role": "system", "content": DATASET_CONTEXT.strip()},
    {"role": "user",   "content": (
        f"Given the following question from PubMedQA:\n"
        f"question: \"{question}\"\n\n"
        f"with these data:\n"
        f"{json.dumps(pubmedqa_example, indent=2)}\n\n"
        f"long_answer: \"{long_answer}\"\n\n"
        f"Please provide the **final_decision**, which must be one of: \"yes\", \"no\", or \"maybe\"."
    )}
]

# — invoke the model as a chat —
result: Conclusion = model(chat_messages[0]['content'], Conclusion)

print("Final decision", result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Final decision no
